In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader,ConcatDataset
import torch.nn as nn
import torch
import torch.optim as optim
import optuna

In [2]:
df = pd.read_csv("/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv")
test = pd.read_csv("/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv")
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
def preprocess(df):
    df = df.copy()
    X = df.drop(columns=['label']).values
    y = df['label'].values
    
    X = X/255.0
    
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
    return X_train,X_test,y_train,y_test

# Custom Dataset and loader

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
class CustomDataset(Dataset):
  def __init__(self,X,y):
    self.features = torch.tensor(X,dtype=torch.float32)
    self.labels = torch.tensor(y,dtype=torch.long)
      
  def __len__(self):
    return len(self.features)
      
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [40]:
X_train,X_test,y_train,y_test = preprocess(df)

train_dataset = CustomDataset(X_train,y_train)
val_dataset = CustomDataset(X_test,y_test)

# Hyperparameter Tuning

In [7]:
class ANN(nn.Module):
    def __init__(self,input_dim,output_dim,no_of_layer,no_of_neuron,dropout_rate):
        super().__init__()
        layers = []
        for i in range(no_of_layer):
            layers.append(nn.Linear(input_dim,no_of_neuron))
            layers.append(nn.BatchNorm1d(no_of_neuron))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = no_of_neuron
            
        layers.append(nn.Linear(input_dim,output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
        return self.model(x)

In [10]:
def objective(trial):
    input_dim = 784
    output_dim = 10
    
    no_of_layer = trial.suggest_int("no_of_layer",1,5)
    no_of_neuron = trial.suggest_int("no_of_neuron",8,128,step=8)
    dropout_rate = trial.suggest_float("dropout_rate",0.1,0.5,step=0.1)
    batch_size = trial.suggest_categorical("batch_size",[32,64,128])
    learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1,log=True)
    optimizer_name = trial.suggest_categorical("optimizer",['sgd','adam','rmsprop'])
    epochs = trial.suggest_int('epochs',10,100,step=10)
    weight_decay = trial.suggest_float('l2-penalty',1e-5,1e-3,log=True)

    model = ANN(input_dim,output_dim,no_of_layer,no_of_neuron,dropout_rate).to(device)

    loss_function = nn.CrossEntropyLoss()
    if optimizer_name=='sgd':
        optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    elif optimizer_name=='adam':
        optimizer = optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
    
    train_dataloader = DataLoader(train_dataset,pin_memory=True,shuffle=True,batch_size=batch_size,num_workers=4)
    val_dataloader = DataLoader(val_dataset,pin_memory=True,shuffle=False,batch_size=batch_size,num_workers=4)

    for epoch in range(epochs):
        model.train()
        for batch_features,batch_labels in train_dataloader:
            batch_features,batch_labels = batch_features.to(device,non_blocking=True),batch_labels.to(device,non_blocking=True)
            
            y_pred = model(batch_features).squeeze(1)
            loss = loss_function(y_pred,batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        total=0
        correct=0
        
        with torch.no_grad():
            for batch_features,batch_labels in val_dataloader:
                batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
                outputs = model(batch_features)
                _, predicted = torch.max(outputs, 1)
                total = total + batch_labels.shape[0]
                correct = correct + (predicted == batch_labels).sum().item()
                
        accuracy = correct/total
        
        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
    return accuracy
            

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=15)

In [28]:
study.best_value*100

88.58333333333334

In [18]:
best_param = study.best_params
best_param

{'no_of_layer': 4,
 'no_of_neuron': 56,
 'dropout_rate': 0.1,
 'batch_size': 64,
 'learning_rate': 0.0006007715179936488,
 'optimizer': 'rmsprop',
 'epochs': 30,
 'l2-penalty': 0.0004388278346115877}

In [35]:
final_model = ANN(
    input_dim=784,
    output_dim=10,
    no_of_layer=best_param['no_of_layer'],
    no_of_neuron=best_param['no_of_neuron'],
    dropout_rate=best_param['dropout_rate']
).to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(
    final_model.parameters(),
    lr=best_param['learning_rate'],
    weight_decay=best_param['l2-penalty']
)

full_train_dataset = ConcatDataset([train_dataset,val_dataset])
full_train_dataloader = DataLoader(
    full_train_dataset,
    batch_size=best_param['batch_size'],
    shuffle=True,
    pin_memory=True,
    num_workers=4,
)

final_model.train()
for epoch in range(best_param['epochs']):
    final_model.train()
    for batch_features,batch_labels in full_train_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        
        # y_pred = final_model(batch_features).squeeze(1)
        y_pred = final_model(batch_features)    
        loss = loss_function(y_pred,batch_labels)
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Evaluation

In [37]:
final_model.eval()

ANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=56, bias=True)
    (1): BatchNorm1d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=56, out_features=56, bias=True)
    (5): BatchNorm1d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=56, out_features=56, bias=True)
    (9): BatchNorm1d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): GELU(approximate='none')
    (11): Dropout(p=0.1, inplace=False)
    (12): Linear(in_features=56, out_features=56, bias=True)
    (13): BatchNorm1d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): GELU(approximate='none')
    (15): Dropout(p=0.1, inplace=False)
    (16): Linear(in_features=56, out_features=10, bias=True)
  )
)

In [43]:
val_dataloader = DataLoader(val_dataset,pin_memory=True,shuffle=False,batch_size=best_param['batch_size'],num_workers=4)
total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in val_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = final_model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"validation accuracy {correct/total}")

validation accuracy 0.922


## ovrefit check

In [46]:
train_dataloader = DataLoader(train_dataset,pin_memory=True,shuffle=False,batch_size=best_param['batch_size'],num_workers=4)


total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in train_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = final_model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"train accuracy {correct/total}")

train accuracy 0.9182916666666666


# no diff file test

In [49]:
test = pd.read_csv("/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv")
test.head(2)

test_X = test.drop(columns=['label'])
test_y = test['label'].values

test_X /= 255.0
test_X = test_X.values

test_dataset = CustomDataset(test_X,test_y)
test_dataloader = DataLoader(test_dataset,shuffle=False,pin_memory=True
                                   ,batch_size=best_param['batch_size']
                                   )

total =0
correct=0

with torch.no_grad():
  for batch_features,batch_labels in test_dataloader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
    y_pred = final_model(batch_features)
    _,predicted = torch.max(y_pred,1)

    total += batch_features.shape[0]
    correct += (predicted==batch_labels).sum().item()

print(f"test accuracy {correct/total}")

test accuracy 0.8837
